# Boundary Experiment — Account A (Final)

**Phase 1 — DLinear baseline:** γ=0.9 only (γ∈{0.0,0.3,0.6} already in `results_leader_follower.csv`), C=21, ρ=0.5, seeds {42,123,456}. 3 runs, ~5 min.
Output: `results_boundary_DLinear_gamma09_acctA.csv`

**Phase 2 — PatchTST CI, P=4:** γ∈{0.6,0.9}, C=21, ρ=0.5, seeds {42,123,456}. 6 runs, ~4.6h.
CD excluded: C×N = 21×255 = 5355 tokens, ~16× P=8 attention cost, exceeds T4 VRAM.
Output: `results_boundary_P4_CI_only_acctA.csv`

Execution order: 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9

In [ ]:
# ── Cell 1: Environment ──────────────────────────────────────────────────────
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Phase 1: DLinear — gamma=0.9 only (other gammas in results_leader_follower.csv)
OUT_PATH_DLINEAR = Path('/kaggle/working/results_boundary_DLinear_gamma09_acctA.csv')
# Phase 2: PatchTST CI at P=4
OUT_PATH_CI      = Path('/kaggle/working/results_boundary_P4_CI_only_acctA.csv')

print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Data generator ───────────────────────────────────────────────────
# Leader-follower VAR(1) process.
# Channels 0–9:   leaders   (pure AR(1), phi=0.8)
# Channels 10–19: followers (each follows leader i with lag-1 coupling gamma)
# Channel 20:     isolate   (pure AR(1), internal negative control)
# Transition matrix A is lower-triangular: spectral radius = phi = 0.8 for all gamma.

N_LEADERS   = 10
N_TOTAL     = 21
PHI         = 0.8
NOISE_STD   = 0.1
N_TIMESTEPS = 20_000


def generate(gamma: float, seed: int) -> np.ndarray:
    """Generate VAR(1) time series of shape (N_TIMESTEPS, N_TOTAL)."""
    rng = np.random.default_rng(seed)
    A   = np.zeros((N_TOTAL, N_TOTAL))
    np.fill_diagonal(A, PHI)
    for i in range(N_LEADERS):
        A[i + N_LEADERS, i] = gamma
    X    = np.zeros((N_TIMESTEPS, N_TOTAL))
    X[0] = rng.standard_normal(N_TOTAL) * NOISE_STD
    noise = rng.standard_normal((N_TIMESTEPS - 1, N_TOTAL)) * NOISE_STD
    for t in range(1, N_TIMESTEPS):
        X[t] = A @ X[t - 1] + noise[t - 1]
    return X


def split_and_normalise(X: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Chronological 70/10/20 split with train-statistics normalisation."""
    T       = len(X)
    n_train = int(T * 0.70)
    n_val   = int(T * 0.10)
    train   = X[:n_train]
    val     = X[n_train:n_train + n_val]
    test    = X[n_train + n_val:]
    mean    = train.mean(axis=0, keepdims=True)
    std     = train.std(axis=0,  keepdims=True) + 1e-8
    return (train - mean) / std, (val - mean) / std, (test - mean) / std

In [ ]:
# ── Cell 3: Configuration ─────────────────────────────────────────────────────
SEEDS    = [42, 123, 456]
LOOKBACK = 512
PRED_LEN = 96

# ── DLinear (Phase 1) ──
# gamma=0.9 only: {0.0, 0.3, 0.6} already exist in results_leader_follower.csv.
# kernel_size=25: standard DLinear setting from Zeng et al. (2022).
# batch_size=128: matches original AR(1) grid and leader-follower experiment.
DLINEAR_GAMMAS    = [0.9]
DLINEAR_BATCH     = 128
DLINEAR_LR        = 1e-4
DLINEAR_MAX_EPOCHS = 50
DLINEAR_PATIENCE  = 10
DLINEAR_KERNEL    = 25   # moving average window

# ── PatchTST CI, P=4 (Phase 2) ──
# gamma={0.6, 0.9}: high-coupling region where CD advantage (if any) should appear.
# CD excluded: seq_len C×N = 21×255 = 5355 tokens, ~16× P=8 attention cost.
CI_GAMMAS    = [0.6, 0.9]
PATCH_SIZE   = 4
STRIDE       = PATCH_SIZE // 2  # = 2
N_PATCHES    = (LOOKBACK - PATCH_SIZE) // STRIDE + 1  # = 255
D_MODEL      = 64
N_HEADS      = 8
N_LAYERS     = 3
DROPOUT      = 0.2
CI_BATCH     = 128
CI_LR        = 1e-4
CI_WARMUP    = 10
CI_MAX_EPOCHS = 50
CI_PATIENCE  = 10

print(f'Phase 1 — DLinear: gamma={DLINEAR_GAMMAS}, {len(DLINEAR_GAMMAS) * len(SEEDS)} runs')
print(f'Phase 2 — CI P={PATCH_SIZE}: gamma={CI_GAMMAS}, {len(CI_GAMMAS) * len(SEEDS)} runs')
print(f'P={PATCH_SIZE}  S={STRIDE}  N_patches={N_PATCHES}  head=Linear({N_PATCHES * D_MODEL}, {PRED_LEN})')

In [ ]:
# ── Cell 4: Models ────────────────────────────────────────────────────────────
class DLinear(nn.Module):
    """Decomposition-Linear model (Zeng et al., 2022).

    Decomposes each variate into trend (moving average) and remainder, then
    applies separate channel-independent linear projections from lookback to
    pred_len. Implementation matches the original AR(1) grid baseline.

    Reference: Zeng et al., 'Are Transformers Effective for Time Series
    Forecasting?', AAAI 2023. https://arxiv.org/abs/2205.13504
    """

    def __init__(self, lookback: int, pred_len: int, n_variates: int, kernel_size: int) -> None:
        super().__init__()
        # Padding preserves sequence length after convolution.
        pad  = (kernel_size - 1) // 2
        self.avg_pool     = nn.AvgPool1d(kernel_size=kernel_size, stride=1, padding=pad)
        self.linear_trend = nn.Linear(lookback, pred_len)
        self.linear_resid = nn.Linear(lookback, pred_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, L, C) → (B, pred_len, C)"""
        # Compute per-channel moving average trend.
        # avg_pool expects (B, C, L); output shape is (B, C, L).
        trend = self.avg_pool(x.permute(0, 2, 1)).permute(0, 2, 1)  # (B, L, C)
        resid = x - trend
        # Apply linear maps channel-independently via broadcasting.
        out_trend = self.linear_trend(trend.permute(0, 2, 1))  # (B, C, pred_len)
        out_resid = self.linear_resid(resid.permute(0, 2, 1))  # (B, C, pred_len)
        return (out_trend + out_resid).permute(0, 2, 1)         # (B, pred_len, C)


class PatchEmbedding(nn.Module):
    def __init__(self, patch_size: int, d_model: int, dropout: float) -> None:
        super().__init__()
        self.proj    = nn.Linear(patch_size, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, N_patches, patch_size) → (B, N_patches, d_model)"""
        return self.dropout(self.proj(x))


class PatchTST_CI(nn.Module):
    """Channel-independent PatchTST. Each variate processed independently.

    Reference: Nie et al., 'A Time Series Is Worth 64 Words', ICLR 2023.
    """

    def __init__(
        self,
        lookback: int,
        pred_len: int,
        patch_size: int,
        stride: int,
        d_model: int,
        n_heads: int,
        n_layers: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.patch_size = patch_size
        self.stride     = stride
        self.n_patches  = (lookback - patch_size) // stride + 1

        self.embed   = PatchEmbedding(patch_size, d_model, dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head    = nn.Linear(self.n_patches * d_model, pred_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, L, C) → (B, pred_len, C)"""
        B, L, C = x.shape
        x_flat  = x.permute(0, 2, 1).reshape(B * C, L)
        patches = x_flat.unfold(dimension=1, size=self.patch_size, step=self.stride)
        enc     = self.encoder(self.embed(patches))         # (B*C, N, d_model)
        out     = self.head(enc.reshape(B * C, -1))         # (B*C, pred_len)
        return out.reshape(B, C, -1).permute(0, 2, 1)       # (B, pred_len, C)


# ── Verify CI head dimensions ─────────────────────────────────────────────────
ci_head = PatchTST_CI(
    LOOKBACK, PRED_LEN, PATCH_SIZE, STRIDE, D_MODEL, N_HEADS, N_LAYERS, DROPOUT
).head
assert ci_head.in_features == N_PATCHES * D_MODEL, \
    f'CI head mismatch: got {ci_head.in_features}, expected {N_PATCHES * D_MODEL}'
assert ci_head.out_features == PRED_LEN, \
    f'CI head out mismatch: got {ci_head.out_features}, expected {PRED_LEN}'
print(f'CI head: {ci_head}  OK')
del ci_head

In [ ]:
# ── Cell 5: Dataset construction ──────────────────────────────────────────────
def make_windows(
    data: np.ndarray, lookback: int, pred_len: int
) -> tuple[torch.Tensor, torch.Tensor]:
    """Slide a window over (T, C) data → (X, Y) float32 tensors."""
    n_windows = len(data) - lookback - pred_len + 1
    X = np.stack([data[i : i + lookback]                       for i in range(n_windows)])
    Y = np.stack([data[i + lookback : i + lookback + pred_len] for i in range(n_windows)])
    return torch.tensor(X, dtype=torch.float32), torch.tensor(Y, dtype=torch.float32)

In [ ]:
# ── Cell 6: Training utilities ────────────────────────────────────────────────
def cosine_lr_with_warmup(
    optimizer: torch.optim.Optimizer,
    epoch: int,
    warmup_epochs: int,
    max_epochs: int,
    base_lr: float,
    min_lr: float = 1e-6,
) -> None:
    """In-place LR update. Linear warmup then cosine decay to min_lr."""
    if epoch < warmup_epochs:
        lr = base_lr * (epoch + 1) / warmup_epochs
    else:
        progress = (epoch - warmup_epochs) / max(1, max_epochs - warmup_epochs)
        lr = min_lr + 0.5 * (base_lr - min_lr) * (1.0 + math.cos(math.pi * progress))
    for pg in optimizer.param_groups:
        pg['lr'] = lr


def _seed_run(seed: int) -> None:
    """Set all RNG seeds for full reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE.type == 'cuda':
        torch.cuda.manual_seed(seed)


def _make_loaders(
    gamma: float, seed: int, batch_size: int
) -> tuple[DataLoader, DataLoader, DataLoader]:
    """Build train/val/test DataLoaders for one (gamma, seed) combination."""
    X_raw = generate(gamma, seed)
    train_data, val_data, test_data = split_and_normalise(X_raw)
    X_tr, Y_tr   = make_windows(train_data, LOOKBACK, PRED_LEN)
    X_val, Y_val = make_windows(val_data,   LOOKBACK, PRED_LEN)
    X_te, Y_te   = make_windows(test_data,  LOOKBACK, PRED_LEN)
    train_loader = DataLoader(
        TensorDataset(X_tr, Y_tr), batch_size=batch_size,
        shuffle=True, drop_last=True,
    )
    val_loader  = DataLoader(TensorDataset(X_val, Y_val), batch_size=batch_size * 4)
    test_loader = DataLoader(TensorDataset(X_te,  Y_te),  batch_size=batch_size * 4)
    return train_loader, val_loader, test_loader


def _train_and_eval(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    test_loader: DataLoader,
    lr: float,
    warmup_epochs: int,
    max_epochs: int,
    patience: int,
    ckpt_path: str,
    use_warmup: bool = True,
) -> tuple[float, float, int, int, int]:
    """Train model, evaluate on test set, return (test_mse, test_mae, best_epoch,
    steps_per_epoch, total_steps_to_best)."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scaler    = GradScaler(DEVICE.type, enabled=(DEVICE.type == 'cuda'))
    criterion = nn.MSELoss()

    best_val      = float('inf')
    best_epoch    = 0
    best_steps    = 0
    patience_cnt  = 0
    total_steps   = 0
    spe           = len(train_loader)

    for epoch in range(max_epochs):
        if use_warmup:
            cosine_lr_with_warmup(optimizer, epoch, warmup_epochs, max_epochs, lr)

        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            with autocast(DEVICE.type, enabled=(DEVICE.type == 'cuda')):
                loss = criterion(model(xb), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            total_steps += 1

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                with autocast(DEVICE.type, enabled=(DEVICE.type == 'cuda')):
                    val_loss += criterion(model(xb), yb).item() * len(xb)
        val_loss /= len(val_loader.dataset)

        if val_loss < best_val:
            best_val     = val_loss
            best_epoch   = epoch + 1
            best_steps   = total_steps
            patience_cnt = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                break

    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
    model.eval()
    test_mse = test_mae = 0.0
    n_elements = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            with autocast(DEVICE.type, enabled=(DEVICE.type == 'cuda')):
                pred = model(xb)
            test_mse   += nn.functional.mse_loss(pred, yb, reduction='sum').item()
            test_mae   += nn.functional.l1_loss(pred,  yb, reduction='sum').item()
            n_elements += yb.numel()
    Path(ckpt_path).unlink(missing_ok=True)  # clean up checkpoint
    return test_mse / n_elements, test_mae / n_elements, best_epoch, spe, best_steps

In [ ]:
# ── Cell 7: Phase 1 — DLinear (gamma=0.9, 3 seeds) ───────────────────────────
# DLinear at gamma={0.0,0.3,0.6} already exists in results_leader_follower.csv.
# Only gamma=0.9 is new. Schema matches results_leader_follower.csv exactly
# (no patch_size column -- DLinear is patch-size independent).

DL_EXPERIMENTS = [(g, s) for g in DLINEAR_GAMMAS for s in SEEDS]

dl_completed: set[tuple] = set()
if OUT_PATH_DLINEAR.exists() and OUT_PATH_DLINEAR.stat().st_size > 100:
    dl_existing = pd.read_csv(OUT_PATH_DLINEAR)
    dl_completed = {(float(r['gamma']), int(r['seed'])) for _, r in dl_existing.iterrows()}
    print(f'DLinear resuming: {len(dl_completed)} runs already complete.')
else:
    print('DLinear starting fresh.')

dl_remaining = [(g, s) for g, s in DL_EXPERIMENTS if (g, s) not in dl_completed]
dl_total     = len(DL_EXPERIMENTS)
dl_done      = len(dl_completed)
print(f'{len(dl_remaining)} DLinear runs remaining of {dl_total}.')

for gamma, seed in dl_remaining:
    dl_done += 1
    label = f'[DLinear {dl_done}/{dl_total}] gamma={gamma} seed={seed}'
    print(f'{label} ...', flush=True)
    t0 = time.time()

    _seed_run(seed)
    train_loader, val_loader, test_loader = _make_loaders(gamma, seed, DLINEAR_BATCH)
    model = DLinear(LOOKBACK, PRED_LEN, N_TOTAL, DLINEAR_KERNEL).to(DEVICE)
    ckpt  = f'/kaggle/working/ckpt_dlinear_g{gamma}_s{seed}.pt'

    test_mse, test_mae, best_epoch, spe, best_steps = _train_and_eval(
        model, train_loader, val_loader, test_loader,
        lr=DLINEAR_LR, warmup_epochs=0, max_epochs=DLINEAR_MAX_EPOCHS,
        patience=DLINEAR_PATIENCE, ckpt_path=ckpt, use_warmup=False,
    )
    elapsed = int(time.time() - t0)
    print(f'{label} test_mse={test_mse:.4f}  best_epoch={best_epoch}  ({elapsed}s)', flush=True)

    row = pd.DataFrame([{
        'dataset':         'leader_follower_var1',
        'C':               N_TOTAL,
        'rho':             0.5,
        'gamma':           gamma,
        'mode':            'DLinear',
        'seed':            seed,
        'test_mse':        round(test_mse, 6),
        'test_mae':        round(test_mae, 6),
        'best_epoch':      best_epoch,
        'batch_size':      DLINEAR_BATCH,
        'steps_per_epoch': spe,
        'total_steps':     best_steps,
    }])
    write_header = not (OUT_PATH_DLINEAR.exists() and OUT_PATH_DLINEAR.stat().st_size > 100)
    row.to_csv(OUT_PATH_DLINEAR, mode='a', header=write_header, index=False)

print(f'\nDLinear done. Saved to {OUT_PATH_DLINEAR}')

In [ ]:
# ── Cell 8: Phase 2 — PatchTST CI, P=4, gamma={0.6, 0.9} ────────────────────
CI_EXPERIMENTS = [(g, s) for g in CI_GAMMAS for s in SEEDS]

ci_completed: set[tuple] = set()
if OUT_PATH_CI.exists() and OUT_PATH_CI.stat().st_size > 100:
    ci_existing  = pd.read_csv(OUT_PATH_CI)
    ci_completed = {(float(r['gamma']), int(r['seed'])) for _, r in ci_existing.iterrows()}
    print(f'CI resuming: {len(ci_completed)} runs already complete.')
else:
    print('CI starting fresh.')

ci_remaining = [(g, s) for g, s in CI_EXPERIMENTS if (g, s) not in ci_completed]
ci_total     = len(CI_EXPERIMENTS)
ci_done      = len(ci_completed)
print(f'{len(ci_remaining)} CI runs remaining of {ci_total}.')

for gamma, seed in ci_remaining:
    ci_done += 1
    label = f'[CI P={PATCH_SIZE} {ci_done}/{ci_total}] gamma={gamma} seed={seed}'
    print(f'{label} ...', flush=True)
    t0 = time.time()

    _seed_run(seed)
    train_loader, val_loader, test_loader = _make_loaders(gamma, seed, CI_BATCH)
    model = PatchTST_CI(
        LOOKBACK, PRED_LEN, PATCH_SIZE, STRIDE, D_MODEL, N_HEADS, N_LAYERS, DROPOUT
    ).to(DEVICE)
    ckpt  = f'/kaggle/working/ckpt_ci_g{gamma}_s{seed}.pt'

    test_mse, test_mae, best_epoch, spe, best_steps = _train_and_eval(
        model, train_loader, val_loader, test_loader,
        lr=CI_LR, warmup_epochs=CI_WARMUP, max_epochs=CI_MAX_EPOCHS,
        patience=CI_PATIENCE, ckpt_path=ckpt, use_warmup=True,
    )
    elapsed = int(time.time() - t0)
    print(f'{label} test_mse={test_mse:.4f}  best_epoch={best_epoch}  ({elapsed}s)', flush=True)

    row = pd.DataFrame([{
        'dataset':         'leader_follower_var1',
        'C':               N_TOTAL,
        'rho':             0.5,
        'gamma':           gamma,
        'patch_size':      PATCH_SIZE,
        'mode':            'CI',
        'seed':            seed,
        'test_mse':        round(test_mse, 6),
        'test_mae':        round(test_mae, 6),
        'best_epoch':      best_epoch,
        'batch_size':      CI_BATCH,
        'steps_per_epoch': spe,
        'total_steps':     best_steps,
    }])
    write_header = not (OUT_PATH_CI.exists() and OUT_PATH_CI.stat().st_size > 100)
    row.to_csv(OUT_PATH_CI, mode='a', header=write_header, index=False)

print(f'\nCI done. Saved to {OUT_PATH_CI}')

In [ ]:
# ── Cell 9: Sanity check and summary ─────────────────────────────────────────
print('=== Phase 1: DLinear (gamma=0.9) ===')
dl_df = pd.read_csv(OUT_PATH_DLINEAR)
print(f'Rows: {len(dl_df)}  (expected {dl_total})')
print(dl_df[['gamma', 'mode', 'seed', 'test_mse', 'best_epoch']].to_string(index=False))

print()
print('=== Phase 2: PatchTST CI, P=4 ===')
ci_df = pd.read_csv(OUT_PATH_CI)
print(f'Rows: {len(ci_df)}  (expected {ci_total})')
print(ci_df[['gamma', 'patch_size', 'mode', 'seed', 'test_mse', 'best_epoch']].to_string(index=False))

print()
print('=== CI mean per gamma ===')
for g, grp in ci_df.groupby('gamma'):
    print(f'  gamma={g}  CI mean={grp["test_mse"].mean():.4f}  std={grp["test_mse"].std():.4f}')

if len(dl_df) < dl_total:
    print(f'WARNING: {dl_total - len(dl_df)} DLinear runs missing.')
if len(ci_df) < ci_total:
    print(f'WARNING: {ci_total - len(ci_df)} CI runs missing.')